# PHASE 6: Model Evaluation, Interpretation & Deployment
**Traceability**
- Issue ID: #6 Model Evaluation, Interpretation & Deployment

## 1. Objectives
- Provide a comprehensive evaluation of all trained models on the test set.
- Interpret the model's decision-making process using SHAP (SHapley Additive exPlanations).
- Quantify prediction uncertainty using Quantile Regression to support maintenance decisions.
- Summarize the final pipeline and provide deployment-ready artifacts.

### 6.1 Import Libraries & Load Artifacts
We load the best models and scalers from previous phases to evaluate them on the unseen test data.

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_squared_error, r2_score, confusion_matrix
import xgboost as xgb
import lightgbm as lgb
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
import shap
import warnings

warnings.filterwarnings('ignore')

# ── Reproducibility Config ──────────────────────────────────────────────
np.random.seed(42)

# ── Global Config ────────────────────────────────────────────────────────
PROCESSED_DIR = Path('../data/processed')
ARTIFACTS_DIR = Path('../artifacts')
FIGURES_DIR = Path('../results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load Data & Artifacts
df_test = pd.read_csv(PROCESSED_DIR / 'test_labeled.csv')
with open(ARTIFACTS_DIR / 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open(ARTIFACTS_DIR / 'best_regressor.pkl', 'rb') as f:
    best_reg = pickle.load(f)

feature_cols = [c for c in df_test.columns if not any(x in c for x in ['unit_number', 'RUL', 'label'])]
X_te = scaler.transform(df_test[feature_cols])
y_te_rul = df_test['RUL'].values

# 2. Define PyTorch Model Architecture (Redefine for evaluation)
class RULPredictorLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim=1):
        super(RULPredictorLSTM, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, output_dim)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out.squeeze()

# 3. Load PyTorch Model and Evaluate (Placeholder: this assumes the model was saved)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pt_model = RULPredictorLSTM(input_dim=len(feature_cols), hidden_dim=64, num_layers=2).to(device)
if os.path.exists('../artifacts/pytorch_lstm.pth'):
    pt_model.load_state_dict(torch.load('../artifacts/pytorch_lstm.pth', map_location=device))
    pt_model.eval()

# 4. Data Preparation for LSTM
SEQ_LEN = 50

def prepare_lstm_input(df, seq_len, scaler, feature_cols):
    """Prepare sequence data for LSTM evaluation."""
    sequences = []
    targets = []
    
    # Scale features first
    df_scaled = df.copy()
    df_scaled[feature_cols] = scaler.transform(df[feature_cols])
    
    for unit in df['unit_number'].unique():
        unit_df = df_scaled[df_scaled['unit_number'] == unit].sort_values('time_cycles')
        
        if len(unit_df) >= seq_len:
            # Take the last sequence of length SEQ_LEN
            seq = unit_df[feature_cols].values[-seq_len:]
            target = unit_df['RUL'].values[-1]
            sequences.append(seq)
            targets.append(target)
            
    return np.array(sequences), np.array(targets)

X_lstm, y_lstm = prepare_lstm_input(df_test, SEQ_LEN, scaler, feature_cols)
X_lstm_tensor = torch.FloatTensor(X_lstm).to(device)

# Get Predictions
with torch.no_grad():
    lstm_preds = pt_model(X_lstm_tensor).cpu().numpy()

# For "Last Cycle" evaluation (ensure alignment)
# Note: Some engines might be shorter than SEQ_LEN and skipped by LSTM
# We filter y_te_last_rul to match LSTM inputs if necessary, but here we assume all test engines > 50 cycles.
y_te_last_rul_lstm = y_lstm

# For "Last Cycle" evaluation
df_test_last = df_test.groupby('unit_number').last().reset_index()
X_te_last = scaler.transform(df_test_last[feature_cols])
y_te_last_rul = df_test_last['RUL'].values

### 6.2 Model Interpretability (SHAP)
Use SHAP values to explain the contribution of each feature to the model's predictions, both globally and for individual engine instances.

In [ ]:
print("\n--- Model Interpretability (SHAP) ---")
explainer = shap.TreeExplainer(best_reg)
shap_values = explainer.shap_values(X_te[:500])

# Global Summary Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_te[:500], feature_names=feature_cols, show=False)
plt.title('SHAP Summary (Global Feature Importance)')
plt.tight_layout()
plt.show()

# Single Engine Waterfall Plot (Near-failure case)
near_failure_idx = np.where(y_te_rul < 30)[0][0]
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap.Explanation(
    values=explainer.shap_values(X_te[near_failure_idx:near_failure_idx+1])[0],
    base_values=explainer.expected_value,
    data=X_te[near_failure_idx],
    feature_names=feature_cols
), show=False)
plt.title(f'SHAP Waterfall (Engine Near Failure, RUL={y_te_rul[near_failure_idx]})')
plt.tight_layout()
plt.show()

### 6.3 Uncertainty Quantification
Train Quantile Regression models to provide an 80% confidence interval for RUL predictions, giving maintenance managers a range of probable outcomes.

In [ ]:
print("\n--- Uncertainty Quantification (Quantile Regression) ---")
df_train = pd.read_csv(PROCESSED_DIR / 'train_labeled.csv')
X_tr = scaler.transform(df_train[feature_cols])
y_tr_rul = df_train['RUL'].values

quantile_models = {}
for q in [0.10, 0.50, 0.90]:
    q_model = lgb.LGBMRegressor(objective='quantile', alpha=q, n_estimators=100, learning_rate=0.05, random_state=42, verbose=-1)
    q_model.fit(X_tr, y_tr_rul)
    quantile_models[q] = q_model

preds_q10 = quantile_models[0.10].predict(X_te_last)
preds_q50 = quantile_models[0.50].predict(X_te_last)
preds_q90 = quantile_models[0.90].predict(X_te_last)

# Visualization of Uncertainty Bands
plt.figure(figsize=(14, 6))
n = 50
plt.fill_between(range(n), preds_q10[:n], preds_q90[:n], alpha=0.3, color='#2E75B6', label='80% Confidence Band (Q10–Q90)')
plt.plot(preds_q50[:n], color='#1F4E79', linewidth=1.5, marker='o', label='Median Prediction (Q50)')
plt.plot(y_te_last_rul[:n], color='#FF7043', linewidth=1.5, linestyle='--', marker='x', label='Actual RUL')
plt.title('RUL Prediction with 80% Uncertainty Bands (Test Set — Last Cycles)')
plt.xlabel('Engine Index'); plt.ylabel('RUL (cycles)'); plt.legend()
plt.tight_layout()
plt.show()

### 6.4 Final Evaluation Summary
Generate a summary of the best model's performance on the test set.

In [ ]:
import time

def nasa_score(y_true, y_pred):
    """NASA asymmetric scoring function."""
    d = y_pred - y_true
    scores = np.where(d >= 0, np.exp(d / 13) - 1, np.exp(-d / 10) - 1)
    return np.sum(scores)

print("\n--- Final Model Comparison Summary ---")

# 1. Collect Predictions & Metrics
models = {}
metrics = []

# XGBoost
t0 = time.time()
xgb_preds = best_reg.predict(X_te_last)
xgb_time = (time.time() - t0) * 1000 / len(X_te_last)  # ms per sample
models['XGBoost'] = (y_te_last_rul, xgb_preds, xgb_time)

# PyTorch LSTM
t0 = time.time()
with torch.no_grad():
    lstm_preds = pt_model(X_lstm_tensor).cpu().numpy()
lstm_time = (time.time() - t0) * 1000 / len(X_lstm_tensor)
models['PyTorch LSTM'] = (y_te_last_rul_lstm, lstm_preds, lstm_time)

# Calculate Metrics
for name, (y_true, y_pred, latency) in models.items():
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = np.mean(np.abs(y_true - y_pred))
    r2 = r2_score(y_true, y_pred)
    nasa = nasa_score(y_true, y_pred)
    metrics.append({
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'NASA Score': nasa,
        'Latency (ms/sample)': latency
    })

metrics_df = pd.DataFrame(metrics)
print(metrics_df.round(4))

# 2. Visual Comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.barplot(x='Model', y='RMSE', data=metrics_df, ax=axes[0,0], palette='viridis')
axes[0,0].set_title('Root Mean Squared Error (Lower is Better)')

sns.barplot(x='Model', y='MAE', data=metrics_df, ax=axes[0,1], palette='viridis')
axes[0,1].set_title('Mean Absolute Error (Lower is Better)')

sns.barplot(x='Model', y='R2', data=metrics_df, ax=axes[1,0], palette='viridis')
axes[1,0].set_title('R² Score (Higher is Better)')

sns.barplot(x='Model', y='NASA Score', data=metrics_df, ax=axes[1,1], palette='viridis')
axes[1,1].set_title('NASA Asymmetric Score (Lower is Better)')

plt.tight_layout()
plt.show()

# 3. Prediction Scatter Plot
plt.figure(figsize=(10, 6))
plt.scatter(y_te_last_rul, xgb_preds, alpha=0.6, label='XGBoost', marker='x')
plt.scatter(y_te_last_rul_lstm, lstm_preds, alpha=0.6, label='LSTM', marker='o')
plt.plot([0, 130], [0, 130], 'r--', label='Perfect Prediction')
plt.xlabel('Actual RUL')
plt.ylabel('Predicted RUL')
plt.title('Actual vs Predicted RUL (Test Set)')
plt.legend()
plt.show()

# 4. Binary Classification Performance (Threshold <= 30)
print("\n--- Binary Classification Comparison (RUL <= 30) ---")
def get_binary_label(rul, threshold=30):
    return (rul <= threshold).astype(int)

y_true_bin = get_binary_label(y_te_last_rul_lstm)
xgb_pred_bin = get_binary_label(xgb_preds)
lstm_pred_bin = get_binary_label(lstm_preds)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(confusion_matrix(y_true_bin, xgb_pred_bin), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('XGBoost Confusion Matrix (Threshold=30)')
axes[0].set_xlabel('Predicted Risk'); axes[0].set_ylabel('Actual Risk')

sns.heatmap(confusion_matrix(y_true_bin, lstm_pred_bin), annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('LSTM Confusion Matrix (Threshold=30)')
axes[1].set_xlabel('Predicted Risk'); axes[1].set_ylabel('Actual Risk')

plt.tight_layout()
plt.show()